# Economy-level ore-sand supply cap (conservative 2050 potential)

**Supply-side ceiling** for the *sand-substitution* CE lever. Ore-sand is fit-for-purpose sand produced as a by-product of metal-ore mining; because it is a by-product of ore that is mined anyway, substituting natural sand with it reduces primary sand extraction **without new mining footprint**. This notebook estimates, for each of the 142 economies, the conservative **maximum annual ore-sand that could be produced** — the physical cap that bounds how much virgin fine sand the substitution measure can displace.

## Method — global potential distributed by mapped mine-area share

$$S_{\text{oresand}}(e) \;=\; g_{2050}\cdot I \cdot A_{\text{maus}}(e),\qquad I=\frac{F_{\text{global}}\cdot r_{\text{sand}}}{A_{\text{maus,global}}}$$

| Symbol | Meaning | Value | Source |
|---|---|---|---|
| `F_global` | global metal-mine tailings (ore-sand feedstock) | **13 Bt/yr** | Golev 2022 **p.82 §10.1** (lower bound: *"total processed and unused ore globally of 13 Bt"*); corroborated Franks 2025 p.1 & Golev 2022 p.10 |
| `r_sand` | fit-for-purpose sand recovered per t tailings | **0.15** | Golev 2022 **p.74 fn19**: *"40% tailings among which 15% is ore-sand"* (= 6% of ore; 1:10 sand:iron-ore) |
| → global ore-sand | `F_global × r_sand` | **~2.0 Bt/yr** | consistent with the *"several billion tonnes per year"* potential (Golev exec. summary) |
| `A_maus(e)` | mapped large-scale mine area in economy *e* | raster | Maus et al. 2022 v2, rasterized in `2-mining-suitability-model/2-rasterize-quarry-area` |
| `g_2050` | future-capacity factor | **1.0** | current mapped mines only; ignores new mines + grade-driven tailings growth → conservative floor |

The conservative 2 Bt/yr global potential is distributed across economies in proportion to their share of global mapped mine area, **using the model's own economy masks** (`results/economy-rasters/`). This replaces the arbitrary fixed substitution % with a physically-grounded, spatially-heterogeneous ceiling.

## How the demand model consumes this

$$\rho_{\text{sub}}(e,t)=\varphi(t)\cdot\lambda\cdot\min\Big[\,a_g(e)\,,\;S_{\text{oresand}}(e)\big/D_{\text{sand}}(e,t)\,\Big]$$

- `a_g(e)`: demand-side **acceptance** ceiling (LIC 5 / LMIC 10 / UMIC 15 / HIC 20 %), from `4-make-CE-parameters`.
- `S_oresand(e)`: this notebook (t/yr).
- `D_sand(e,t)`: fine-aggregate (sand) demand from IMAGINE-Concrete (`fine_to_binder` streams).

## Output
`data/oresand-supply-cap-142.csv` — one row per economy: `maus_area_km2`, `area_share`, `oresand_supply_kt_yr`, `income_group`.

In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
RES = '1km'   # allocation resolution (matches the Maus raster + economy mask)

## Step 1 — load the Maus mine-area raster and the economy mask

Both are on the shared Eckert IV grid, so they align cell-for-cell. The economy raster stores an integer `value` per economy (1:1 with `economies_142` in `pseudo-codes-economies-142.csv`).

In [3]:
maus_area = rio.open(RESULTS / 'mining-area' / f'maus-mine-area-eck4-{RES}.tif').read(1)
economy   = rio.open(RESULTS / 'economy-rasters' / f'economy-raster-eck4-{RES}.tif').read(1)
assert maus_area.shape == economy.shape, 'grid mismatch between Maus raster and economy mask'

codes = pd.read_csv(RESULTS / 'pseudo-codes-economies-142.csv')
val2econ = codes.drop_duplicates('value').set_index('value')['economies_142'].to_dict()
all_econ = sorted(codes['economies_142'].unique())
print(f'Maus raster total: {maus_area.sum():,.0f} km^2 | economies in list: {len(all_econ)}')

Maus raster total: 101,810 km^2 | economies in list: 142


## Step 2 — parameters and the global ore-sand intensity

`F_global` and `r_sand` are the two literature anchors (see table above). Their product is the conservative global ore-sand potential; dividing by total mapped mine area gives the intensity `I` (t ore-sand per km² of mine per year).

In [4]:
F_GLOBAL_T = 13e9      # global metal-mine tailings, t/yr  (Golev 2022 p.82; Franks 2025 p.1)
R_SAND     = 0.15      # fit-for-purpose ore-sand per t tailings  (Golev 2022 p.74 fn19)
G_2050     = 1.0       # future-capacity factor (1.0 = current mapped mines; conservative)

ORE_SAND_GLOBAL_T = F_GLOBAL_T * R_SAND          # ~2.0 Bt/yr
A_GLOBAL_KM2 = float(maus_area.sum())            # global mapped mine area (denominator)
I = ORE_SAND_GLOBAL_T / A_GLOBAL_KM2             # t ore-sand / km^2 / yr
print(f'global ore-sand potential : {ORE_SAND_GLOBAL_T/1e9:.2f} Bt/yr')
print(f'global mapped mine area   : {A_GLOBAL_KM2:,.0f} km^2')
print(f'intensity I               : {I:,.0f} t ore-sand / km^2 / yr')

global ore-sand potential : 1.95 Bt/yr
global mapped mine area   : 101,810 km^2
intensity I               : 19,153 t ore-sand / km^2 / yr


## Step 3 — sum mine area by economy and distribute the global potential

Sum the Maus per-cell area within each economy mask, map the raster `value` to its `economies_142` code, then `S_oresand(e) = g · I · A_maus(e)`. Economies with no mapped mining receive 0.

In [5]:
mask = economy > 0
vals, inv = np.unique(economy[mask], return_inverse=True)
area_by_val = np.bincount(inv, weights=maus_area[mask])

df = pd.DataFrame({'value': vals, 'maus_area_km2': area_by_val})
df['economy'] = df['value'].map(val2econ)
df = (df.groupby('economy', as_index=False)['maus_area_km2'].sum())

# reindex to ALL 142 economies (0 for those with no mapped mining)
df = df.set_index('economy').reindex(all_econ).fillna(0.0).reset_index()

df['area_share'] = df['maus_area_km2'] / A_GLOBAL_KM2
df['oresand_supply_kt_yr'] = G_2050 * I * df['maus_area_km2'] / 1000.0

# attach income group (from the CE-parameters table) for cross-reference
ce = pd.read_csv(DATA / 'CE-parameters-142.csv')[['economy', 'income_group']]
df = df.merge(ce, on='economy', how='left')
df = df.sort_values('oresand_supply_kt_yr', ascending=False).reset_index(drop=True)

## Step 4 — validate

In [6]:
area_inside = df['maus_area_km2'].sum()
print(f"area inside 142 economies : {area_inside:,.0f} km^2  "
      f"({100*area_inside/A_GLOBAL_KM2:.1f}% of global)")
print(f"modelled ore-sand supply  : {df['oresand_supply_kt_yr'].sum()/1e6:.3f} Bt/yr "
      f"(= {ORE_SAND_GLOBAL_T/1e9:.2f} Bt x inside-fraction)")
print(f"economies with supply > 0 : {(df['oresand_supply_kt_yr']>0).sum()} of 142\n")
top = df.head(15).copy()
top['oresand_Mt_yr'] = top['oresand_supply_kt_yr'] / 1e3
print(top[['economy','income_group','maus_area_km2','area_share','oresand_Mt_yr']]
      .to_string(index=False,
          formatters={'maus_area_km2':'{:,.0f}'.format,'area_share':'{:.3f}'.format,
                      'oresand_Mt_yr':'{:,.1f}'.format}))

area inside 142 economies : 101,593 km^2  (99.8% of global)
modelled ore-sand supply  : 1.946 Bt/yr (= 1.95 Bt x inside-fraction)
economies with supply > 0 : 120 of 142

economy income_group maus_area_km2 area_share oresand_Mt_yr
    SUN         UMIC        15,514      0.152         297.1
    CHN         UMIC        10,388      0.102         199.0
    AUS          HIC         8,482      0.083         162.5
    USA          HIC         8,204      0.081         157.1
    IDN         LMIC         8,010      0.079         153.4
    BRA         UMIC         5,929      0.058         113.6
    CAN          HIC         5,093      0.050          97.5
    CHL          HIC         4,567      0.045          87.5
    ZAF         UMIC         3,597      0.035          68.9
    PER         UMIC         3,547      0.035          67.9
    GUY         UMIC         2,391      0.023          45.8
    ARG         UMIC         2,311      0.023          44.3
    IND         LMIC         2,295      0.023     

## Step 5 — save

In [7]:
out = DATA / 'oresand-supply-cap-142.csv'
df.to_csv(out, index=False)
print(f'Saved {out}  ({len(df)} economies)')
df.head(10)

Saved d:\wty\Documents\GitHub\aggregate-mining-biodiversity\data\oresand-supply-cap-142.csv  (142 economies)


,economy,maus_area_km2,area_share,oresand_supply_t_yr,income_group
0,SUN,15514.013998,0.152382,2.971439e+08,UMIC
1,CHN,10387.728001,0.102030,1.989589e+08,UMIC
2,AUS,8482.339002,0.083315,1.624644e+08,HIC
3,USA,8204.386002,0.080585,1.571407e+08,HIC
4,IDN,8009.937999,0.078675,1.534164e+08,LMIC
5,BRA,5928.966001,0.058235,1.135590e+08,UMIC
6,CAN,5092.696001,0.050021,9.754173e+07,HIC
7,CHL,4567.409002,0.044862,8.748077e+07,HIC
8,ZAF,3597.375003,0.035334,6.890146e+07,UMIC
9,PER,3546.885999,0.034838,6.793443e+07,UMIC


## Notes, conservatism and sensitivity

- **Conservative by construction**: `F_global` = 13 Bt/yr is an explicit *lower bound* (Golev 2022 p.82); `r_sand` = 0.15 is Golev's own iron-ore allocation figure (p.74 fn19), not a max-recovery value; `g_2050` = 1.0 ignores both new mines and the grade-decline-driven rise in tailings (Franks 2025 cite 858 Bt tailings+waste-rock for copper alone, 2020–50).
- **Sensitivity knobs**: raise `r_sand` to 0.30–0.40 (improved recovery / coarse-flotation and pre-sorting; Golev shows upgrading to 99.5% silica is feasible) and/or `g_2050` > 1 for an upper envelope.
- **Known bias**: Maus polygons carry no commodity tag, so distributing metal-derived tailings by *all* mapped mine area slightly over-allocates coal-heavy economies (e.g. IDN, parts of CHN/USA). Because the demand-side ceiling `a_g` usually binds first in those economies (via the `min()`), this rarely changes results; a commodity overlay (USGS MRDS / Jasansky 2023) is the clean upgrade if a reviewer asks.
- **Economy aggregations** (e.g. former-USSR → `SUN`) are handled automatically because we allocate with the model's own economy masks.